In [20]:
import pandas as pd

DATA_PATH = "hotel_bookings.csv"

# ---------------------------------------------------------------------------
# Step 1: Identify missing / invalid values
# ---------------------------------------------------------------------------
def identify_missing_invalid(df: pd.DataFrame) -> dict:
    """Scan the dataset for:
        - missing values (nulls) per column
        - exact duplicate rows
        - columns that SHOULD be numeric but contain text (a sign of
          formatting problems, e.g. blank strings, stray spaces)
        - inconsistent category values (same value written differently:
          spacing, casing, etc.)
    Returns a dict summary so the results can be reused.
    """

    print("\n" + "=" * 70)
    print("STEP 1: IDENTIFY MISSING / INVALID VALUES")
    print("=" * 70)

    findings = {}

# --- Missing values ---
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0].sort_values(ascending=False)
    findings["missing_values"] = nulls.to_dict()
    print("\nMissing values per column (columns with 0 nulls are omitted):")
    print(nulls if not nulls.empty else "  None found")

    # --- Exact duplicate rows ---
    dup_count = df.duplicated().sum()
    findings["duplicate_rows"] = int(dup_count)
    print(f"\nExact duplicate rows: {dup_count}")

    # --- Columns expected to be numeric but read as text ---
    expected_numeric_cols = [
        "Lead Time",
        "Stays in Weekend Nights",
        "Stays in Week Nights",
        "Adults",
        "Children",
        "Average Daily Rate",
        "Parking Spaces",
    ]
    non_numeric = {}
    for col in expected_numeric_cols:
        if col not in df.columns:
            continue
        # try to coerce to numeric; anything that WAS non-null but becomes
        # NaN after coercion is an invalid/malformed value (typos, units
        # mixed into the text, stray words, etc.)
        coerced = pd.to_numeric(df[col], errors="coerce")
        was_present = df[col].notna()
        bad_mask = was_present & coerced.isna()
        bad_count = int(bad_mask.sum())
        if bad_count > 0:
            sample_bad_values = sorted(set(df.loc[bad_mask, col].astype(str)))[:8]
            non_numeric[col] = {
                "invalid_count": bad_count,
                "example_values": sample_bad_values,
            }
    findings["numeric_columns_with_invalid_text"] = non_numeric
    print("\nExpected-numeric columns that contain non-numeric / malformed values:")
    for col, info in non_numeric.items():
        print(f"  {col}: {info['invalid_count']} value(s) can't be converted "
              f"to a number, e.g. {info['example_values']}")

    # --- Inconsistent category text (casing / spacing) ---
    category_cols = ["Hotel", "Meal", "Deposit Type", "Customer Type",
                      "Market Segment", "Distribution Channel",
                      "Reserved Room Type", "Assigned Room Type"]
    inconsistencies = {}
    for col in category_cols:
        if col not in df.columns:
            continue
        raw_values = df[col].dropna().unique()
        # normalize by stripping spaces and lowercasing, then see if that
        # collapses multiple raw spellings into one "true" category
        normalized_map = {}
        for v in raw_values:
            key = str(v).strip().lower()
            normalized_map.setdefault(key, set()).add(v)
        variants = {k: v for k, v in normalized_map.items() if len(v) > 1}
        if variants:
            inconsistencies[col] = variants
    findings["inconsistent_category_values"] = inconsistencies
    print("\nColumns with inconsistent spelling/casing/spacing (same value, "
          "written multiple ways):")
    for col, variants in inconsistencies.items():
        print(f"\n  {col}:")
        for normalized, raw_forms in variants.items():
            print(f"    '{normalized}' appears as: {sorted(raw_forms)}")

    return findings


# ---------------------------------------------------------------------------
# Step 2: Define cleaning rules
# ---------------------------------------------------------------------------
def define_cleaning_rules() -> list:
    """Document the cleaning rules the team has agreed on, based on the
    findings from Step 1. This function does not clean anything yet — it
    just returns the plan as a list of rule descriptions, so it can be
    printed, saved to the report, and used as a checklist for whoever
    implements data_prep.py in the next user story.
    """

    rules = [
        "1. Drop exact duplicate rows (df.drop_duplicates()).",
        "2. Standardize text categories: strip leading/trailing whitespace "
        "and use a consistent case (e.g. Title Case for 'Hotel', "
        "upper-case codes for 'Meal') so 'City Hotel', ' city hotel', "
        "'CITY HOTEL' all map to one canonical value.",
        "3. Convert numeric-looking text columns (Lead Time, Adults, "
        "Children, Stays in Weekend/Week Nights, Average Daily Rate, "
        "Parking Spaces) to proper numeric dtypes. Before coercing, strip "
        "out embedded units/currency text first (e.g. '1 guests' -> 1, "
        "'0 spaces' -> 0, '$ 90' / 'USD 85.00' -> 90 / 85.00), then apply "
        "pd.to_numeric(..., errors='coerce') for anything left invalid "
        "(e.g. the literal string 'unknown').",
        "4. Handle missing values per column: "
        "numeric columns (Adults, Children, Lead Time, ADR, Parking "
        "Spaces) -> fill with median or 0 where appropriate; "
        "categorical columns (Meal, Country, Market Segment, "
        "Distribution Channel) -> fill with 'Unknown' or the most "
        "frequent category, and document the choice.",
        "5. Treat placeholder/undefined text values ('Undefined', "
        "'unknown', blank strings) in categorical columns as missing "
        "before applying rule 4.",
        "6. Validate logical ranges: rows where Adults + Children == 0 "
        "(no guests) or Average Daily Rate < 0 are invalid and should be "
        "flagged or removed.",
        "7. Convert Arrival Date and Reservation Status Date to proper "
        "datetime dtype (pd.to_datetime(..., errors='coerce')).",
        "8. Keep an audit trail: log/print how many rows were affected by "
        "each rule so the cleaning step is transparent and reproducible.",
    ]

    print("\n" + "=" * 70)
    print("STEP 2: DEFINE CLEANING RULES")
    print("=" * 70)
    for rule in rules:
        print(f"  {rule}")

    return rules


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------
def main():
    df = pd.read_csv(DATA_PATH)
    identify_missing_invalid(df)  # Step 1
    define_cleaning_rules()        # Step 2


if __name__ == "__main__":
    main()


STEP 1: IDENTIFY MISSING / INVALID VALUES

Missing values per column (columns with 0 nulls are omitted):
Country                    166
Children                    81
Meal                        80
Distribution Channel        71
Average Daily Rate          68
Market Segment              56
Stays in Weekend Nights     28
Lead Time                   28
Parking Spaces              24
Stays in Week Nights        24
Adults                      19
Reservation Status Date      5
dtype: int64

Exact duplicate rows: 204

Expected-numeric columns that contain non-numeric / malformed values:
  Lead Time: 32 value(s) can't be converted to a number, e.g. ['unknown']
  Stays in Weekend Nights: 17 value(s) can't be converted to a number, e.g. ['0 nights', '1 nights', '2 nights', '3 nights']
  Stays in Week Nights: 19 value(s) can't be converted to a number, e.g. ['0 nights', '1 nights', '2 nights', '3 nights', '4 nights', '5 nights']
  Adults: 25 value(s) can't be converted to a number, e.g. ['1 gue

In [ ]:

import pandas as pd
import numpy as np

# ---------------------------------------------------------------------------
# Step 3: Clean the dataset
# ---------------------------------------------------------------------------
def clean_hotel_bookings(file_path=DATA_PATH, output_path='hotel_bookings_cleaned.csv'):
    """Apply the data-cleaning rules to the hotel bookings dataset. Step 3."""
    df = pd.read_csv(file_path).copy()
    original_rows = len(df)

    # 1) Remove exact duplicates
    df = df.drop_duplicates().copy()
    print(f"Rows after dropping duplicates: {len(df)} (removed {original_rows - len(df)})")

    # 2) Normalize common text categories
    text_cols = [
        'Hotel', 'Meal', 'Country', 'Deposit Type', 'Customer Type',
        'Market Segment', 'Distribution Channel', 'Reserved Room Type',
        'Assigned Room Type', 'Reservation Status'
    ]
    for col in text_cols:
        if col not in df.columns:
            continue
        df[col] = df[col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
        df[col] = df[col].replace({'': pd.NA, 'nan': pd.NA, 'NaN': pd.NA, 'unknown': pd.NA, 'undefined': pd.NA})
        if col == 'Hotel':
            df[col] = df[col].str.title()
        elif col == 'Meal':
            df[col] = df[col].str.upper()
        else:
            df[col] = df[col].str.title()
        df[col] = df[col].fillna('Unknown')

    # 3) Numeric columns: convert strings to numbers and fill missing values
    numeric_cols = [
        'Lead Time', 'Stays in Weekend Nights', 'Stays in Week Nights',
        'Adults', 'Children', 'Average Daily Rate', 'Required Car Parking Spaces',
        'Booking Changes', 'Previous Cancellations', 'Previous Bookings Not Canceled',
        'Days in Waiting List', 'Total of Special Requests'
    ]
    for col in numeric_cols:
        if col not in df.columns:
            continue
        df[col] = df[col].astype(str)
        df[col] = df[col].str.replace(r'[$,%\s]', '', regex=True)
        df[col] = pd.to_numeric(df[col], errors='coerce')
        median_value = df[col].median()
        if pd.isna(median_value):
            median_value = 0
        df[col] = df[col].fillna(median_value)

    # 4) Date columns
    for col in ['Reservation Status Date']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

    # 5) Validate logical rules
    if 'Adults' in df.columns and 'Children' in df.columns:
        guest_total = df['Adults'].fillna(0) + df['Children'].fillna(0)
        invalid_guests = guest_total == 0
        print(f"Rows with no adults and no children: {int(invalid_guests.sum())}")
        df = df.loc[~invalid_guests].copy()

    if 'Average Daily Rate' in df.columns:
        invalid_adr = df['Average Daily Rate'] < 0
        print(f"Rows with negative ADR values: {int(invalid_adr.sum())}")
        df = df.loc[~invalid_adr].copy()

    # 6) Final missing-value check
    print("\nMissing values after cleaning:")
    print(df.isnull().sum()[df.isnull().sum() > 0].to_string())

    # 7) Save cleaned data
    df.to_csv(output_path, index=False)
    print(f"\nCleaned dataset saved to: {output_path}")
    print(f"Final shape: {df.shape}")
    return df

cleaned_df = clean_hotel_bookings()
cleaned_df.head()


Rows after dropping duplicates: 25500 (removed 204)
Rows with no adults and no children: 42
Rows with negative ADR values: 2

Missing values after cleaning:
Parking Spaces              24
Reservation Status Date    378

Cleaned dataset saved to: hotel_bookings_cleaned.csv
Final shape: (25456, 23)


,Booking ID,Hotel,Is Canceled,Lead Time,Arrival Date,Stays in Weekend Nights,Stays in Week Nights,Adults,Children,Meal,...,Is Repeated Guest,Reserved Room Type,Assigned Room Type,Deposit Type,Customer Type,Average Daily Rate,Parking Spaces,Special Requests,Reservation Status,Reservation Status Date
0,1,City Hotel,0,14.0,2015-09-02,2.0,5.0,2.0,0.0,BB,...,0,D,D,No Deposit,Transient,74.61,0,2,Check-Out,2015-09-09
1,2,City Hotel,1,307.0,2015-08-20,0.0,2.0,2.0,0.0,BB,...,1,A,A,No Deposit,Transient-Party,62.80,0,0,Canceled,2014-10-17
2,3,Resort Hotel,0,101.0,2016-12-15,0.0,2.0,1.0,0.0,BB,...,0,A,D,No Deposit,Transient,25.64,0,3,Check-Out,2016-12-17
3,4,City Hotel,0,21.0,2017-01-30,1.0,0.0,1.0,0.0,BB,...,1,A,A,No Deposit,Transient,80.00,0,1,Check-Out,2017-01-31
4,5,City Hotel,1,206.0,2016-08-28,2.0,5.0,2.0,0.0,BB,...,0,A,A,No Deposit,Transient,99.21,0,1,Canceled,2016-02-23


In [ ]:
import pandas as pd

CLEAN_DATA_PATH = "hotel_bookings_clean.csv"

# Columns expected to be numeric after cleaning
NUMERIC_COLS = [
    "Lead Time",
    "Stays in Weekend Nights",
    "Stays in Week Nights",
    "Adults",
    "Children",
    "Average Daily Rate",
    "Parking Spaces",
]

# Columns expected to be datetime after cleaning
DATE_COLS = ["Arrival Date", "Reservation Status Date"]

# Categorical columns that should have one consistent spelling per value
CATEGORY_COLS = [
    "Hotel", "Meal", "Deposit Type", "Customer Type",
    "Market Segment", "Distribution Channel",
    "Reserved Room Type", "Assigned Room Type",
]

# Raw placeholder text that should NOT remain in the cleaned dataset
RAW_PLACEHOLDERS = {"undefined", ""}


def verify_cleaning(df: pd.DataFrame) -> tuple:
    """Run pass/fail checks against a cleaned DataFrame.
    Returns (all_passed: bool, report: dict)."""

    report = {}

    # 1. No exact duplicate rows
    dup_count = int(df.duplicated().sum())
    report["no_duplicate_rows"] = {
        "passed": dup_count == 0,
        "detail": f"{dup_count} duplicate row(s) remaining",
    }

    # 2. Numeric columns are actually numeric dtype
    for col in NUMERIC_COLS:
        if col not in df.columns:
            continue
        report[f"{col}_is_numeric"] = {
            "passed": pd.api.types.is_numeric_dtype(df[col]),
            "detail": f"dtype is {df[col].dtype}",
        }

    # 3. Date columns are actually datetime dtype
    for col in DATE_COLS:
        if col not in df.columns:
            continue
        report[f"{col}_is_datetime"] = {
            "passed": pd.api.types.is_datetime64_any_dtype(df[col]),
            "detail": f"dtype is {df[col].dtype}",
        }

    # 4. No leftover raw placeholder text ("undefined" / blank) in categories
    leftover = 0
    for col in CATEGORY_COLS:
        if col not in df.columns:
            continue
        leftover += int(
            df[col].astype(str).str.strip().str.lower().isin(RAW_PLACEHOLDERS).sum()
        )
    report["no_raw_placeholder_text_left"] = {
        "passed": leftover == 0,
        "detail": f"{leftover} raw placeholder value(s) remaining",
    }

    # 5. Category columns: only one spelling per value 
    remaining_variants = {}
    for col in CATEGORY_COLS:
        if col not in df.columns:
            continue
        normalized_map = {}
        for v in df[col].dropna().unique():
            key = str(v).strip().lower()
            normalized_map.setdefault(key, set()).add(v)
        variants = {k: v for k, v in normalized_map.items() if len(v) > 1}
        if variants:
            remaining_variants[col] = variants
    report["no_category_spelling_variants"] = {
        "passed": len(remaining_variants) == 0,
        "detail": remaining_variants if remaining_variants else "clean",
    }

    # 6. No logically invalid rows: zero guests, or negative rate
    zero_guest_rows = 0
    if "Adults" in df.columns and "Children" in df.columns:
        zero_guest_rows = int(((df["Adults"].fillna(0) + df["Children"].fillna(0)) == 0).sum())
    negative_rate_rows = 0
    if "Average Daily Rate" in df.columns:
        negative_rate_rows = int((df["Average Daily Rate"] < 0).sum())
    report["no_invalid_guest_or_rate_rows"] = {
        "passed": zero_guest_rows == 0 and negative_rate_rows == 0,
        "detail": f"{zero_guest_rows} zero-guest row(s), "
                  f"{negative_rate_rows} negative-rate row(s)",
    }

    # 7. No unexpected remaining nulls in the key columns
    key_cols = NUMERIC_COLS + ["Meal", "Country", "Market Segment", "Distribution Channel"]
    remaining_nulls = {
        col: int(df[col].isna().sum())
        for col in key_cols if col in df.columns and df[col].isna().sum() > 0
    }
    report["no_remaining_nulls_in_key_columns"] = {
        "passed": len(remaining_nulls) == 0,
        "detail": remaining_nulls if remaining_nulls else "clean",
    }

    all_passed = all(check["passed"] for check in report.values())
    return all_passed, report


def print_verification_report(all_passed: bool, report: dict) -> None:
    print("=" * 70)
    print("CLEANING VERIFICATION REPORT")
    print("=" * 70)
    for check_name, result in report.items():
        status = "PASS" if result["passed"] else "FAIL"
        print(f"  [{status}] {check_name} -> {result['detail']}")
    print("-" * 70)
    print(f"OVERALL: {'ALL CHECKS PASSED' if all_passed else 'SOME CHECKS FAILED'}")
    print("=" * 70)


def main():
    df = pd.read_csv(CLEAN_DATA_PATH, parse_dates=DATE_COLS)
    print(f"Loaded '{CLEAN_DATA_PATH}': {df.shape[0]} rows x {df.shape[1]} columns\n")

    all_passed, report = verify_cleaning(df)
    print_verification_report(all_passed, report)


if __name__ == "__main__":
    main()